# Ground, Singlet, and Triplet States for H2 using FCI

In [ ]:
#makes a file containing all the energy for each bond length

# h2_fci_pec.py
import psi4
import numpy as np
import csv

#choose basis (small to large):
#cc-pVDZ, cc-pVTZ, cc-pVQZ, cc-pV5Z,
#aug-cc-pVDZ, aug-cc-pVTZ, aug-cc-pVQZ, aug-cc-pV5Z"

basis = "aug-cc-pVQZ"

n_singlet_roots = 11              #set number of single roots (includes ground state)
n_triplet_roots = 11              #set number of triplet roots

r_min, r_max, n_points = 0.2, 5.0, 1000          #max and min bond length, number of points
csv_out = "h2_fci_singlet_triplet.csv"           #name csv output file
psi4_out = "h2_fci_spin_scan.out"                #name psi4 output file


psi4.set_output_file(psi4_out, False)            #(Names output file, overrides existing files)
psi4.set_memory("4 GB")                          #How much ram psi4 is allowed to use
psi4.set_num_threads(4)                          #number of threads to use


#create an array of points
bond_lengths = np.linspace(r_min, r_max, n_points)

#run scan for each root at a set R
def run_h2_fci(R, multiplicity, roots, basis):
    psi4.core.clean()                #removes scratch files, clears data
    psi4.core.clean_options()        #resets options

    mol = psi4.geometry(f"""
    0 {multiplicity}                 #charge, multiplicity
    H                                #first hydrogen at (0,0,0)
    H 1 {R}                          #second hydrogen at distance R from hydrogen #1
    units angstrom                   #units of R in angstroms
    symmetry c1                      #turns off symmetry
    """)

    #set reference based on multiplicity
    reference = "rhf" if multiplicity == 1 else "rohf"
    
    psi4.set_options({
        "basis": basis,              #set basis
        "reference": reference,      #set reference   
        "guess": "sad",              #initial guess
        "scf_type": "pk",            #compute exact integrals
        "e_convergence": 1e-10,      #convergence limits
        "d_convergence": 1e-10,
        "maxiter": 200,              #max iterations
        "num_roots": roots
    })

    psi4.energy("fci", molecule=mol)      #calculate energy using Full Configuration Interaction for my mol
    vars_dict = psi4.core.variables()     #creates a dictionary with psi4 variables

    energies = []                         #create an empty list for energies
    
    #add each calculated root to the energies list
    for n in range(roots):
        energies.append(vars_dict[f"CI ROOT {n} TOTAL ENERGY"])

    #returns a list of the energies of each root at the n values
    return energies

#empty list for rows
rows = []


#run psi4 for each R
for R in bond_lengths:
    singlet_energies = [np.nan] * n_singlet_roots                     #set up blank lists to hold lists of energies for each singlet root
    triplet_energies = [np.nan] * n_triplet_roots                     #set up blank lists to hold lists of energies for each triplet root

    try:
        singlet_energies = run_h2_fci(R, 1, n_singlet_roots, basis)   #run function for singlet roots for each R
        #print(f"R = {R:.3f} singlet done")                           #print when done
    except Exception as err:                                          #If it fails
        print(f"R = {R:.3f} singlet failed: {err}")                   #print if it fails

    try:
        triplet_energies = run_h2_fci(R, 3, n_triplet_roots, basis)   #run function for triplet roots for each R
        #print(f"R = {R:.3f} triplet done")                           #print when done
    except Exception as err:                                          #If it fails
        print(f"R = {R:.3f} triplet failed: {err}")                   #print if it fails

    row = [R] + singlet_energies + triplet_energies                   #make a list of ([R] + [singlet energies] + [triplet energies])
    rows.append(row)                                                  #add R and all energies to rows list


#create headers
headers = ["R_angstrom"]                                              #list for headers, starting with R
headers += [f"E_singlet_root_{i}" for i in range(n_singlet_roots)]    #add a new header for each singlet root
headers += [f"E_triplet_root_{i}" for i in range(n_triplet_roots)]    #add a new header for each triplet root

with open(csv_out, "w", newline="") as f:
    w = csv.writer(f)                                #make csv writer
    w.writerow(headers)                              #make row with header
    w.writerows(rows)                                #add data to csv file                           #add data to csv file

print("Done. Wrote h2_fci_singlet_triplet.csv")


    try:
        energies = run_h2plus_fci(R, roots, basis)   #runs function for R
        rows.append([R] + energies)                  #adds ([R], [energies]) to my list of rows
        #print(f"R = {R:.3f} done")                  #print when done
    except Exception as err:                         #If it fails
        print(f"R = {R:.3f} failed: {err}")          #print if it fails
        rows.append([R] + [np.nan] * roots)          #leave a gap for failure
    finally:
        psi4.core.clean()
        psi4.core.clean_options()